In [1]:
import datetime
import time
import tarfile

import boto3
import pandas as pd
import numpy as np
# from sagemaker import get_execution_role
import sagemaker
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml

from sagemaker.core.helper.session_helper import Session, get_execution_role


sm_boto3 = boto3.client("sagemaker")

sess = Session()

role = get_execution_role()

region = sess.boto_session.region_name

bucket = sess.default_bucket()  # this could also be a hard-coded bucket name

print("Using bucket " + bucket)



sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Using bucket sagemaker-us-east-1-155361516759


In [2]:
# we use the Boston housing dataset
data = fetch_openml(name='boston', version=1, as_frame=True)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, random_state=42
)

trainX = pd.DataFrame(X_train, columns=data.feature_names)
trainX["target"] = y_train

testX = pd.DataFrame(X_test, columns=data.feature_names)
testX["target"] = y_test



In [4]:
trainX.head()

trainX.to_csv("boston_train.csv")
testX.to_csv("boston_test.csv")

# send data to S3. SageMaker will take training data from s3
trainpath = sess.upload_data(
    path="boston_train.csv", bucket=bucket, key_prefix="sagemaker/sklearncontainer"
)

testpath = sess.upload_data(
    path="boston_test.csv", bucket=bucket, key_prefix="sagemaker/sklearncontainer"
)



In [5]:
%%writefile script.py

import argparse
import joblib
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor


# inference functions ---------------
def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir, "model.joblib"))
    return clf


if __name__ == "__main__":

    print("extracting arguments")
    parser = argparse.ArgumentParser()

    # hyperparameters sent by the client are passed as command-line arguments to the script.
    # to simplify the demo we don't use all sklearn RandomForest hyperparameters
    parser.add_argument("--n-estimators", type=int, default=10)
    parser.add_argument("--min-samples-leaf", type=int, default=3)

    # Data, model, and output directories
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    parser.add_argument("--train-file", type=str, default="boston_train.csv")
    parser.add_argument("--test-file", type=str, default="boston_test.csv")
    parser.add_argument(
        "--features", type=str
    )  # in this script we ask user to explicitly name features
    parser.add_argument(
        "--target", type=str
    )  # in this script we ask user to explicitly name the target

    args, _ = parser.parse_known_args()

    print("reading data")
    train_df = pd.read_csv(os.path.join(args.train, args.train_file))
    test_df = pd.read_csv(os.path.join(args.test, args.test_file))

    print("building training and testing datasets")
    X_train = train_df[args.features.split()]
    X_test = test_df[args.features.split()]
    y_train = train_df[args.target]
    y_test = test_df[args.target]

    # train
    print("training model")
    model = RandomForestRegressor(
        n_estimators=args.n_estimators, min_samples_leaf=args.min_samples_leaf, n_jobs=-1
    )

    model.fit(X_train, y_train)

    # print abs error
    print("validating model")
    abs_err = np.abs(model.predict(X_test) - y_test)

    # print couple perf metrics
    for q in [10, 50, 90]:
        print("AE-at-" + str(q) + "th-percentile: " + str(np.percentile(a=abs_err, q=q)))

    # persist model
    path = os.path.join(args.model_dir, "model.joblib")
    joblib.dump(model, path)
    print("model persisted at " + path)
    print(args.min_samples_leaf)



Overwriting script.py


In [6]:
# Optional local smoke test only.
# This model.joblib is not used for deployment.
! python script.py --n-estimators 100 \
                   --min-samples-leaf 2 \
                   --model-dir ./ \
                   --train ./ \
                   --test ./ \
                   --features 'CRIM ZN INDUS CHAS NOX RM AGE DIS RAD TAX PTRATIO B LSTAT' \
                   --target target

extracting arguments
reading data
building training and testing datasets
training model
validating model
AE-at-10th-percentile: 0.34870238095237843
AE-at-50th-percentile: 1.5401297619047725
AE-at-90th-percentile: 4.445312051282058
model persisted at ./model.joblib
2


In [12]:
# We use the Estimator from the SageMaker Python SDK
# from sagemaker.sklearn.estimator import SKLearn  <- deprecated


# FRAMEWORK_VERSION = "0.23-1"

# sklearn_estimator = SKLearn(  <- deprecated : check code blocks below for updated SageMaker v3 API utilization
#     entry_point="script.py",
#     role=get_execution_role(),
#     instance_count=1,
#     instance_type="ml.m4.xlarge",#"ml.c5.xlarge",#"ml.m4.xlarge",
#     framework_version=FRAMEWORK_VERSION,
#     base_job_name="rf-scikit",
#     metric_definitions=[{"Name": "median-AE", "Regex": "AE-at-50th-percentile: ([0-9.]+).*$"}],
#     hyperparameters={
#         "n-estimators": 100,
#         "min-samples-leaf": 3,
#         "features": "CRIM ZN INDUS CHAS NOX RM AGE DIS RAD TAX PTRATIO B LSTAT",
#         "target": "target",
#     },
# )

# import time
# tic = time.time()

# # launch training job, with asynchronous call
# sklearn_estimator.fit({"train": trainpath, "test": testpath}, wait=False)

# toc = time.time()
# print(toc - tic)


In [13]:
#  TODO: Set up debugging and profiling rules and hooks
# The hooks and rules now live inside train.py; this cell prepares the job
import os
import shutil
import boto3
from sagemaker.core import image_uris


data_s3 = f"s3://{bucket}/sagemaker/sklearncontainer/data"
output_s3 = f"s3://{bucket}/sagemaker/sklearncontainer/model"

# Dedicated source dir: using "." would upload data/dogImages (~1.1 GB) along with the code.
# train.py stays at the project root (used by the local debugging block) and is copied here.
source_dir = "scripts"
os.makedirs(source_dir, exist_ok=True)
shutil.copy("script.py", f"{source_dir}/script.py")

# smdebug is no longer needed; tensorboard + psutil are
with open(f"{source_dir}/requirements.txt", "w") as f:
    f.write("psutil\n")

training_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.9-0",
    py_version="py3",
    image_scope="training",
    instance_type="ml.m5.large"
)

print(role)
print(data_s3)
print(output_s3)
print(training_image)

arn:aws:iam::155361516759:role/service-role/AmazonSageMaker-ExecutionRole-20261023T161945
s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/data
s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/model
683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.9-0-cpu-py3


In [14]:
!python -m py_compile scripts/script.py && echo "OK"

OK


In [15]:
from sagemaker.core.shapes import MetricDefinition

base_job_name="rf-scikit"

# metric_definitions=[{"Name": "median-AE", "Regex": "AE-at-50th-percentile: ([0-9.]+).*$"}]
metric_definitions = [
    MetricDefinition(
        name="median-AE",
        regex="AE-at-50th-percentile: ([0-9.]+).*$",
    )
]

hyperparameters={
        "n-estimators": 100,
        "min-samples-leaf": 3,
        "features": "CRIM ZN INDUS CHAS NOX RM AGE DIS RAD TAX PTRATIO B LSTAT",
        "target": "target",
    }
print(metric_definitions)

print(hyperparameters)

[MetricDefinition(name='median-AE', regex='AE-at-50th-percentile: ([0-9.]+).*$')]
{'n-estimators': 100, 'min-samples-leaf': 3, 'features': 'CRIM ZN INDUS CHAS NOX RM AGE DIS RAD TAX PTRATIO B LSTAT', 'target': 'target'}


In [16]:
# TODO: Create and fit an estimator
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    Compute,
    InputData,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)

trainer = ModelTrainer(
    training_image=training_image,
    role=role,
    sagemaker_session=sess,
    base_job_name=base_job_name,
    source_code=SourceCode(
        source_dir=source_dir,
        requirements="requirements.txt",
        entry_script="script.py",
    ),
    compute=Compute(instance_type="ml.m5.large", instance_count=1),
    output_data_config=OutputDataConfig(s3_output_path=output_s3),
    hyperparameters=hyperparameters,
    stopping_condition=StoppingCondition(max_runtime_in_seconds=3600)
).with_metric_definitions(metric_definitions)

[10/24/26 06:23:02] INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=12418562;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=12418563;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=12418570;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=12418571;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-lea                     
                             rn:1.9-0-cpu-py3                                                                      

In [17]:
data_input = f"s3://{bucket}/sagemaker/sklearncontainer/"

trainer.train(
    input_data_config=[
        InputData(channel_name="train", data_source=data_input),
        InputData(channel_name="test", data_source=data_input),
    ],
    wait=True
) # TODO: Remember to include your data channels

[10/24/26 06:23:11] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=12418578;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=12418579;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#325\325]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


                    INFO     Creating training_job resource.                                     ]8;id=12418586;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12418587;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31296\31296]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=12418594;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=12418595;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#350\350]8;;\

Output()

...

                    INFO     rf-scikit-20261024062311/algo-1-1792823034:                         ]8;id=12419201;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12419202;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31642\31642]8;;\
                             model persisted at /opt/ml/model/model.joblib                                         

                    INFO     rf-scikit-20261024062311/algo-1-1792823034:                         ]8;id=12419207;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12419208;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31642\31642]8;;\
                             3                                                                                     

                    INFO     rf-scikit-20261024062311/algo-1-1792823034:                         ]8;id=12419213;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12419214;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31642\31642]8;;\
                             Training Container Execution Completed                                                

                    INFO     rf-scikit-20261024062311/algo-1-1792823034:                         ]8;id=12419219;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12419220;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31642\31642]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

[10/24/26 06:25:20] INFO     Final Resource Status: Completed                                    ]8;id=12419226;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12419227;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31648\31648]8;;\

In [24]:
# Create the inference folder and script
import os

# Create a folder dedicated to endpoint inference code.
inference_dir = "./inference"
os.makedirs(inference_dir, exist_ok=True)

inference_script = """
import joblib
import os


def model_fn(model_dir):
    # SageMaker extracts model.tar.gz into model_dir.
    # model.joblib was saved to SM_MODEL_DIR during training.
    model_path = os.path.join(model_dir, "model.joblib")
    return joblib.load(model_path)
"""

with open(os.path.join(inference_dir, "inference.py"), "w") as file:
    file.write(inference_script)

print("Created:", os.path.join(inference_dir, "inference.py"))

Created: ./inference/inference.py


In [25]:
# Deprecated code block : check code block below for updated deployment code
# sklearn_estimator.latest_training_job.wait(logs="None")
# artifact = sm_boto3.describe_training_job(
#     TrainingJobName=sklearn_estimator.latest_training_job.name
# )["ModelArtifacts"]["S3ModelArtifacts"]

# print("Model artifact persisted at " + artifact)

# from sagemaker.sklearn.model import SKLearnModel

# model = SKLearnModel(
#     model_data=artifact,
#     role=get_execution_role(),
#     entry_point="script.py",
#     framework_version=FRAMEWORK_VERSION,
# )

# predictor = model.deploy(instance_type="ml.c5.large", initial_instance_count=1)



In [26]:
import boto3

from sagemaker.core import image_uris
from sagemaker.core.training.configs import SourceCode
from sagemaker.serve import ModelBuilder
from sagemaker.serve.utils.types import ModelServer


# ---------------------------------------------------------
# 1. Retrieve the model.tar.gz produced by the training job
# ---------------------------------------------------------

training_job_name =  trainer._latest_training_job.training_job_name

sm_client = boto3.client("sagemaker", region_name=region)

training_job_description = sm_client.describe_training_job(
    TrainingJobName=training_job_name
)

model_artifact_s3_uri = training_job_description[
    "ModelArtifacts"
]["S3ModelArtifacts"]

print("Model artifact:", model_artifact_s3_uri)



Model artifact: s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/model/rf-scikit-20261024062311/output/model.tar.gz


In [27]:
# Repacking
import os
import shutil
import tarfile

s3 = boto3.client("s3")
src_bucket, src_key = model_artifact_s3_uri.replace("s3://", "").split("/", 1)
s3.download_file(src_bucket, src_key, "model.tar.gz")

shutil.rmtree("repack", ignore_errors=True)
os.makedirs("repack/code")
with tarfile.open("model.tar.gz") as t:
    t.extractall("repack", filter="data")                      # -> repack/model.pth
shutil.copy("inference/inference.py", "repack/code/inference.py")  # -> repack/code/inference.py

with tarfile.open("model_with_code.tar.gz", "w:gz") as t:
    for name in os.listdir("repack"):
        t.add(os.path.join("repack", name), arcname=name)

# Sanity check: should list model.pth and code/inference.py
with tarfile.open("model_with_code.tar.gz") as t:
    print(t.getnames())

model_with_code_url = sess.upload_data(
    path="model_with_code.tar.gz", bucket=bucket, key_prefix=f"sagemaker/sklearncontainer/model_deploy"
)
print(f"Repacked model artifact: {model_with_code_url}")

['code', 'code/inference.py', 'model.joblib']
Repacked model artifact: s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/model_deploy/model_with_code.tar.gz


In [28]:
# sm_client.create_model(
#     ModelName=model_name,
#     ExecutionRoleArn=role,
#     PrimaryContainer={
#         "Image": inference_image_uri,
#         "ModelDataUrl": repacked_model_s3_uri,
#         "Environment": {
#             "SAGEMAKER_PROGRAM": "inference.py",

#             # The container downloads this archive and finds
#             # code/inference.py inside it.
#             "SAGEMAKER_SUBMIT_DIRECTORY": repacked_model_s3_uri,

#             "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
#             "SAGEMAKER_REGION": region,
#         },
#     },
# )

from sagemaker.serve import ModelBuilder

# predictor=trainer.deploy(initial_instance_count=1, instance_type="ml.m5.large")  # TODO: Add your deployment configuration like instance type and number of instances

inference_image_uri = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.9-0",
    py_version="py3",
    image_scope="inference",
    instance_type="ml.m5.large"
)

builder = ModelBuilder(
    model_path=model_with_code_url,
    image_uri=inference_image_uri,
    role_arn=role,
    env_vars={
        "SAGEMAKER_PROGRAM": "inference.py", 
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code"
    },
)
model = builder.build()

model

[10/24/26 06:44:24] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=12419299;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=12419300;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#340\340]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=12419305;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=12419306;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#374\374]8;;\

                    DEBUG    No ModelMetadata provided. ModelBuilder is not handling    ]8;id=12419311;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=12419312;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1335\1335]8;;\
                             MLflow model input                                                                    

                    INFO     Creating model with name: model-81f75eb5                        ]8;id=12419317;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=12419318;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#2029\2029]8;;\

                    INFO     ✅ Model has been created: 'model-81f75eb5' using server None in ]8;id=12419323;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=12419324;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#3357\3357]8;;\
                             SAGEMAKER_ENDPOINT mode (ARN:                                                         
                             arn:aws:sagemaker:us-east-1:155361516759:model/model-81f75eb5)                        

Model(model_name='model-81f75eb5', primary_container=ContainerDefinition(container_hostname=Unassigned(), image='683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.9-0-cpu-py3', image_config=Unassigned(), mode='SingleModel', model_data_url='s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/model_deploy/model_with_code.tar.gz', model_data_source=ModelDataSource(s3_data_source=S3ModelDataSource(s3_data_type='S3Object', compression_type='Gzip', s3_uri='s3://sagemaker-us-east-1-155361516759/sagemaker/sklearncontainer/model_deploy/model_with_code.tar.gz', model_access_config=Unassigned(), hub_access_config=Unassigned(), manifest_s3_uri=Unassigned(), e_tag=Unassigned(), manifest_etag=Unassigned())), additional_model_data_sources=Unassigned(), environment={'SAGEMAKER_CONTAINER_LOG_LEVEL': '20', 'SAGEMAKER_PROGRAM': 'inference.py', 'SAGEMAKER_REGION': 'us-east-1', 'SAGEMAKER_SUBMIT_DIRECTORY': '/opt/ml/model/code'}, model_package_name=Unassigned(), inference_s

In [29]:
predictor = builder.deploy(initial_instance_count=1, instance_type="ml.m5.large")

[10/24/26 06:44:44] INFO     Creating endpoint-config with name endpoint-649c686b            ]8;id=12419329;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=12419330;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1017\1017]8;;\

[10/24/26 06:44:45] INFO     Creating endpoint with name endpoint-649c686b                   ]8;id=12419335;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=12419336;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1232\1232]8;;\

                    WARNING  Failed to enable live logging: An error occurred                ]8;id=12419341;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=12419342;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#2951\2951]8;;\
                             (AccessDeniedException) when calling the FilterLogEvents                              
                             operation: User:                                                                      
                             arn:aws:sts::155361516759:assumed-role/AmazonSageMaker-Executio                       
                             nRole-20261023T161945/SageMaker is not authorized to perform:                         
                             logs:FilterLogEvents on resource:                                                     
                             arn:aws:logs:us-east-1:155361516759:log-group:/aws/sagemaker/En                       
                             dpoints/endpoint-649c686b because no identity-based policy                            
                             allows the logs:FilterLogEvents action. Fallback to default                           
                             logging...                                                                            

Output()

[10/24/26 06:47:46] INFO     ✅ Deployment successful: Endpoint 'endpoint-649c686b' using     ]8;id=12419348;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=12419349;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#2687\2687]8;;\
                             None in SAGEMAKER_ENDPOINT mode (ARN:                                                 
                             arn:aws:sagemaker:us-east-1:155361516759:endpoint/endpoint-649c6                      
                             86b)                                                                                  

In [44]:
# print(predictor.predict(testX[data.feature_names]))
test_sample = testX[data.feature_names]
# Convert the DataFrame to CSV expected by the sklearn container
payload = test_sample.to_csv(
    index=False,
    header=False,
)

response = predictor.invoke(
    body=payload,
    content_type="text/csv",
)

result = response.body.read().decode("utf-8")
print(result)

[22.92940133477634, 32.35111284271285, 16.956851623376625, 23.80317251082251, 17.47371742424242, 21.662118795093793, 19.460928860028858, 15.99065966810966, 21.297963888888884, 20.965192865467866, 20.21158124098124, 20.0527858946609, 7.7234079365079324, 21.8568886002886, 19.763349927849934, 25.898860389610398, 18.685474603174608, 8.241441269841268, 44.26356396658896, 15.392466269841263, 24.102525793650795, 24.04799274891774, 14.892748953823954, 23.320077403152403, 14.684659126984126, 15.585209523809525, 21.38959354256856, 14.33330855810856, 19.886694444444434, 21.37307464757465, 20.15224191919192, 23.617995923520922, 29.239396897546893, 20.285659523809525, 14.532542027417028, 16.239601010101016, 34.66916005661006, 19.417366211566215, 21.119345634920634, 23.937565043290046, 19.44982056277057, 29.221408530358513, 44.45607933455433, 19.58782786935287, 22.47897355699856, 13.835338023088024, 15.53547586580087, 24.263888528138526, 18.898463600288597, 29.226742424242435, 20.930895895770906, 33

In [45]:
#sm_boto3.delete_endpoint(EndpointName=predictor.endpoint_name)
predictor.delete()
print("Endpoint successfully deleted.")

[10/24/26 07:01:32] INFO     Deleting Endpoint - endpoint-649c686b                               ]8;id=12419355;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12419356;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#10427\10427]8;;\

Endpoint successfully deleted.
